**Indexing**
1. Load the pdf data using PyMyPDFLoader from langchain_community.document_loaders
2. Split the text/docs into chunks using Recursive... from langchain_textsplitters
3. Generate text embeddings with:
    - OpenAI text-embedding-3-small
    - Testing with ollama:
      Embedding model: ollama pull hf.co/CompendiumLabs/bge-base-en-v1.5-gguf
      Language model: ollama pull hf.co/bartowski/Llama-3.2-1B-Instruct-GGUF
4. Store embeddings in a vector database:
    - Pinecone
    - FAISS
    - Chromadb

**Retrieval**
1. Translate users question to embeddings
2. Retrieve relevant chunks according to the question as context
3. Add a system prompt (a prompt for the llm's behavior), the context and the user's question (again)
4. Push this result to the llm to generate a grounded answer


Load .env variables

In [ ]:
import os

OPENAI_API_KEY = os.getenv("OPENAI_API_KEYS")

### Load PDF

In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader

loader = PyMuPDFLoader(file_path="../data/admission_requirement.pdf")
docs = loader.load()
print(docs)

### Split PDF into chunks

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

doc_splitter = RecursiveCharacterTextSplitter(
  chunk_size=500,
  chunk_overlap=100
)
chunked_docs = doc_splitter.split_documents(docs)

print(f"Number of chunks: {len(chunked_docs)}")

for i, chunk in enumerate(chunked_docs, start=1):
  print(f"\nChunk {i}:")
  print(f"metadata: {chunk.metadata}\n")
  print(f"content: '{chunk.page_content[:150]}'")


### Translate Chunks into embeddings

In [ ]:
from langchain_openai import OpenAIEmbeddings

embedding_function = OpenAIEmbeddings(
  model="text-embedding-3-small"
)

In [ ]:
# from langchain_chroma import Chroma

# chroma_vector_db = Chroma.from_documents(
#   documents=chunked_docs,
#   embedding=embedding_function,
#   persist_directory="./chroma_vector_db"
# )
# print(f"Chroma Vector Database with {chroma_vector_db._collection.count()} embeddings")

In [ ]:
from langchain_chroma import Chroma

chroma_vector_db = Chroma(
  embedding_function=embedding_function,
  persist_directory="./chroma_vector_db"
)
print(f"Chroma Vector Database with {chroma_vector_db._collection.count()} embeddings")


### Retrieval

In [ ]:
# Store into vectordb
retriever = chroma_vector_db.as_retriever(
  search_kwargs={"k":2}
)

retrieved_docs = retriever.invoke("What is the cutoff point Sociology")
print(retrieved_docs)

### Generation

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
  model="gpt-4o-mini"
)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

system_prompt = """You are KNUST Admissions Assistant, an AI agent that helps prospective students find admission requirements and information for programs at Kwame Nkrumah University of Science and Technology (KNUST).Answer ONLY from the provided context. If the answer is not in the context, say you don't have that information. Cite the chunk numbers you used, e.g. [1].""
"""

prompt = ChatPromptTemplate.from_messages([
      ("system", system_prompt + "\n\nRetrived context:\n{context}"),
      ("human", "{question}"),
  ])


In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()} | prompt | llm |StrOutputParser()
)

response = chain.invoke("What's the requirement for reading Sociology")
print(response)